In [1]:
import pandas as pd
import numpy as np


import os, wrds

os.environ["WRDS_USERNAME"] = "YOUR_WRDS_USERNAME"

db = wrds.Connection(wrds_username = os.environ["WRDS_USERNAME"])
print(" Successful")

Loading library list...
Done
 Successful


너가 지금 하고 싶은게 뭐야?

나는 지금 분기별로 발표했을때마다 그때의 per와 시간이 지날 수록의 per에 대해 궁금해

예를 들어 3분기 실적발표 이후에 실적에 따라서 새로운 eps를 도입을 할꺼야 시장은, 그리고 다음 eps가 나올때까지 어떻게 가격이 움직이냐~ 이걸 보고싶은거야

실적 발표가 된 다음 per과 시간이 흐를 수록 per이 어떻게 되는지가 궁금한거야 

근데 지금 이걸 다 parquet으로 가져와서 하는건 너무 비효율저이야 

이걸 SQL 쿼리를 써서 한꺼번에 가져오는걸 해보자 

어떤 회사를 기준으로 할꺼야? -> APPLE

그러면 필요한 정보가 어떤거야?

funda 에서는
gvkey, rdq(실적 발표날), datadate(분기말 날짜 3분기면 9/30)
prccq(분기말 주가), niq(분기 순이익), cshoq(주식수), ajexq(주식 병합 분할 조정값), epsfxq(분기 eps)


crsp.dsf (일별 주가)
permno, date, prc, shrout, ,cfacpr, cfacshr 

crsp.dsenames (주식명 티커)
permno, ticker, comnam


In [ ]:
df_fund = db.raw_sql("""
            # funda 데이터
            WITH fund AS ( 
                SELECT 
                    f.gvkey, f.rdq, f.datadate, f.prrcq, f.niq, f.cshoq, f.ajexq, f.epsfxq,
                    
                FROM
                    FROM comp.fundq f
                WHERE f.indfmt = 'INDL'
                AND f.datafmt = 'STD'
                AND f.consol = 'C'
                AND f.popsrc = 'D'
                AND gvkey = '001690'
                     
            ),
            
            fund_ccm AS (
                SELECT 
                     f.*,
                     l.permno AS permno
                     l.linkdt::date,
                     COALESCE(l.linkenddt::date, DATE '2099-12-31') AS linkenddt,
                FROM fund f
                JOIN crsp.ccmxpf_linktable AS l
                ON l.gvkey = f.gvkey
                AND linkprim IN ('LU','LC')
                AND l.linkprim IN ('P', 'C')
                AND lpermno IS NOT NULL
                AND f.datadate BETWEEN l.linkdt AND COALESCE(l.linkenddt, DATE '2099-12-31')
            ),
                     
            eps_ttm AS (
            -- 분할 조정 EPS(분기)를 합산해서 TTM (최근 4분기)
                     
                SELECT
                     gvkey, permno, datadate, rdq,
                     niq, cshoq, prccq,
                     
                     -- 분할 조정 EPS(분기)
                     (epsfxq/ NULLIF(ajexq,0)) AS eps_adj_q,
                     --TTM EPS (최근 4개 분기 합)
                     SUM(epsfxq/ NULL(ajexq,0)) OVER (
                     PARTITION BY gvkey
                     ORDER BY datadate
                     ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
                     ) AS eps_ttm_adj
                     FROM fund_ccm
            ),
                     

            eps_line AS (
                    -- 다음 rdq(유효기간 끝) 계산
                     
                    SELECT 
                    gvkey, permno, datadate, rdq, niq, cshoq, prccq, eps_adj_q, eps_ttm_adj,
                    LEAD(rdq) OVER (PARTITION BY permno ORDER BY rdq) AS next_rdq
                    FROM eps_ttm
            ),
                     
            -- 필요하면 CRSP 범위를 줄여서 성능 개선
            dsf_cut AS (
                    SELECT d.permno, d.date::date, d.prc, d.cfacpr
                    FROM crsp.dsf d
                    WHERE d.shrcd IN (10,11) AND d.exchrd IN (1,2,3)
            ),
            
            result AS (
                SELECT
                    e.gvkey, e.permno, )


"""")

In [25]:
columns_stock = ["permno","date","prc","shrout","cfacpr","cfacshr","adj_prc","adj_shrout","ticker","comnam"]

stocks= df_stock[columns_stock]

columns_fund = ["gvkey","datadate","rdq","niq","cshoq","prccq","ajexq","epsfxq"]

funda = df_fund[columns_fund]


In [26]:
df_ccm

,gvkey,permno,linktype,linkprim,linkdt,linkenddt
0,001000,25881.0,LU,P,1970-11-13,1978-06-30
1,001001,10015.0,LU,P,1983-09-20,1986-07-31
2,001002,10023.0,LC,C,1972-12-14,1973-06-05
3,001003,10031.0,LU,C,1983-12-07,1989-08-16
4,001004,54594.0,LU,P,1972-04-24,2099-12-31
...,...,...,...,...,...,...
32844,352262,23773.0,LC,P,2023-03-17,2099-12-31
32845,353444,23209.0,LC,P,2022-07-22,2099-12-31
32846,355398,25134.0,LC,P,2024-05-17,2099-12-31
32847,356128,24704.0,LC,C,2024-01-19,2099-12-31
